In [1]:
%pylab inline

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [40]:
from pycbc.waveform.waveform import _mode_array_map
from pycbc.waveform import get_fd_waveform
from pycbc.filter import overlap_cplx, sigma

In [72]:
approx = "IMRPhenomXHM"
delta_f=0.125
f_lower=20
fh = 512
params = dict(
    mass1=100,
    mass2=10,
    inclination=pi/2,
    delta_f=delta_f,
    f_lower=f_lower,
    f_final=fh,
    approximant=approx
)

In [73]:
lm = ["22", "44"]
hp, hc = get_fd_waveform(**params, mode_array=_mode_array_map(lm, approx))
abs((overlap_cplx(hp, hc)).imag)

0.9925232112445065

In [67]:
from pycbc.psd import aLIGOZeroDetHighPower

In [74]:
length = int(fh / delta_f) + 1
psd = aLIGOZeroDetHighPower(length, delta_f, f_lower)

# check orthogonalization routine

In [75]:
phase = 0
h_dom, _ = get_fd_waveform(**params, coa_phase = phase, mode_array=_mode_array_map("22", approx))
h_sub, _ = get_fd_waveform(**params, coa_phase = phase, mode_array=_mode_array_map("32", approx))

sigma_dom = sigma(h_dom, psd, f_lower, fh)
sigma_sub = sigma(h_sub, psd, f_lower, fh)

if sigma_dom > 0:
    if sigma_sub > 0:
        # generate the orthogonal waveform
        zeta = overlap_cplx(h_dom, h_sub, psd, f_lower, fh, normalized=False)
        zeta = zeta / sigma_dom / sigma_sub
        norm = 1 / (sqrt(1 - abs(zeta) ** 2))
        h_sub_perp = (h_sub / sigma_sub - zeta * h_dom / sigma_dom) * norm
        sigma_sub_perp = sigma_sub * norm
for phase2 in linspace(0,pi/2,10):
    h_dom, _ = get_fd_waveform(**params, coa_phase = phase2, mode_array=_mode_array_map("22", approx))
    print(abs(overlap_cplx(h_sub_perp, h_dom, psd, f_lower, fh)), abs(zeta), sigma_sub_perp, sigma_sub)

1.8585795227214515e-17 0.6716875409591202 2009.5717358477702 1488.7601954683457
2.222949841039214e-17 0.6716875409591202 2009.5717358477702 1488.7601954683457
4.767248533286248e-17 0.6716875409591202 2009.5717358477702 1488.7601954683457
4.9260643200906254e-17 0.6716875409591202 2009.5717358477702 1488.7601954683457
2.8805703737907246e-17 0.6716875409591202 2009.5717358477702 1488.7601954683457
6.645302041370679e-17 0.6716875409591202 2009.5717358477702 1488.7601954683457
2.0199018144765765e-17 0.6716875409591202 2009.5717358477702 1488.7601954683457
3.751898158764304e-17 0.6716875409591202 2009.5717358477702 1488.7601954683457
4.2013175029278966e-17 0.6716875409591202 2009.5717358477702 1488.7601954683457
4.512364981913592e-17 0.6716875409591202 2009.5717358477702 1488.7601954683457
